# Attention on Sets: Gravitational Force Prediction

A set of point masses in 2D. Each has position $(x, y)$ and mass $m$. The task: predict the net gravitational force vector on each particle, given all the others.

This is a **set-to-set** task. We train a minimal transformer and visualize what the attention heads learn.

We **mask out self-attention** (each particle can only attend to *other* particles), which forces the model to use cross-element communication and produces interpretable attention patterns.

In [ ]:
from functools import partial

import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np

## 1. Data: Random Point Masses and Gravitational Forces

In [ ]:
def generate_batch(key, batch_size, n_particles, softening=0.1):
    """Generate random point masses and compute gravitational forces.

    Each particle has features [x, y, mass].
    Target is the net force [Fx, Fy] on each particle.
    """
    k1, k2 = jr.split(key)

    # Random positions in [-1, 1]^2 and masses in [0.5, 2.0]
    pos = jr.uniform(k1, (batch_size, n_particles, 2), minval=-1.0, maxval=1.0)
    mass = jr.uniform(k2, (batch_size, n_particles, 1), minval=0.5, maxval=2.0)

    # Features: [x, y, mass]
    inputs = jnp.concatenate([pos, mass], axis=-1)  # (batch, n, 3)

    # Compute pairwise gravitational forces
    # F_ij = G * m_i * m_j * (r_j - r_i) / |r_j - r_i|^3
    # We set G = 1
    dx = pos[:, None, :, :] - pos[:, :, None, :]  # (batch, n, n, 2) -- r_j - r_i
    dist = jnp.sqrt(jnp.sum(dx**2, axis=-1, keepdims=True) + softening**2)  # (batch, n, n, 1)

    m_i = mass[:, :, None, :]  # (batch, n, 1, 1)
    m_j = mass[:, None, :, :]  # (batch, 1, n, 1)

    # Force from j on i
    F_ij = m_i * m_j * dx / dist**3  # (batch, n, n, 2)

    # Zero out self-interaction
    eye = jnp.eye(n_particles)[None, :, :, None]
    F_ij = F_ij * (1.0 - eye)

    # Net force on each particle
    forces = jnp.sum(F_ij, axis=2)  # (batch, n, 2)

    return inputs, forces

In [ ]:
# Visualize one example
key = jr.PRNGKey(42)
inputs, forces = generate_batch(key, 1, 6)
pos = inputs[0, :, :2]
mass = inputs[0, :, 2]
F = forces[0]

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.scatter(pos[:, 0], pos[:, 1], s=mass * 200, c="steelblue", edgecolors="k", zorder=5)
ax.quiver(pos[:, 0], pos[:, 1], F[:, 0], F[:, 1], color="firebrick", scale=30, width=0.006, zorder=4)
for i in range(len(pos)):
    ax.annotate(f"{i}", (pos[i, 0] + 0.05, pos[i, 1] + 0.05), fontsize=10, color="gray")
ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-1.3, 1.3)
ax.set_aspect("equal")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Point masses and gravitational forces")
plt.tight_layout()
plt.show()

## 2. Minimal Set Transformer

No positional encoding or causal mask, but attention to itself is masked out.

In [ ]:
import optax
from flax import linen as nn
from flax.training import train_state


class SetTransformer(nn.Module):
    """Minimal set transformer: no positional encoding, no causal mask.
    Self-attention masked out so each element must attend to others."""

    d_model: int = 64
    d_mlp: int = 128
    n_layers: int = 2
    n_heads: int = 2
    d_head: int = 16
    d_out: int = 2  # force vector

    @nn.compact
    def __call__(self, x, return_attention=False):
        # x: (n, d_in)
        n = x.shape[0]
        attn_maps = []

        # Mask: prevent self-attention (diagonal -> -inf)
        mask = jnp.where(jnp.eye(n), -1e9, 0.0)

        # Project input features to model dimension
        x = nn.Dense(self.d_model)(x)

        for _ in range(self.n_layers):
            # Pre-norm
            x1 = nn.LayerNorm()(x)

            # Multi-head self-attention
            heads = []
            layer_attn = []
            for _ in range(self.n_heads):
                q = nn.Dense(self.d_head)(x1)  # (n, d_head)
                k = nn.Dense(self.d_head)(x1)
                v = nn.Dense(self.d_head)(x1)

                score = q @ k.T / jnp.sqrt(self.d_head) + mask  # (n, n)
                attn = jax.nn.softmax(score, axis=-1)
                heads.append(attn @ v)
                layer_attn.append(attn)

            # Concat + project
            x_attn = jnp.concatenate(heads, axis=-1)
            x_attn = nn.Dense(self.d_model)(x_attn)

            # Residual
            x = x + x_attn
            attn_maps.append(jnp.stack(layer_attn))  # (n_heads, n, n)

            # Pre-norm + MLP
            x2 = nn.LayerNorm()(x)
            x2 = nn.Dense(self.d_mlp)(x2)
            x2 = jax.nn.gelu(x2)
            x2 = nn.Dense(self.d_model)(x2)

            # Residual
            x = x + x2

        # Final projection to force vector
        x = nn.LayerNorm()(x)
        x = nn.Dense(self.d_out)(x)

        if return_attention:
            return x, attn_maps
        return x

## 3. Training

In [ ]:
n_particles = 6
model = SetTransformer()

# Initialize
key = jr.PRNGKey(0)
dummy = jnp.ones((n_particles, 3))
params = model.init(key, dummy)

n_params = sum(p.size for p in jax.tree.leaves(params))
print(f"Parameters: {n_params:,}")

In [ ]:
# Loss: MSE on force vectors
def loss_fn(params, inputs, targets):
    # vmap over batch
    preds = jax.vmap(lambda x: model.apply(params, x))(inputs)
    return jnp.mean((preds - targets) ** 2)


@jax.jit
def train_step(state, inputs, targets):
    loss, grads = jax.value_and_grad(loss_fn)(state.params, inputs, targets)
    state = state.apply_gradients(grads=grads)
    return state, loss


# Optimizer
schedule = optax.warmup_cosine_decay_schedule(
    init_value=1e-5, peak_value=3e-3, warmup_steps=200, decay_steps=8000, end_value=1e-5
)
optimizer = optax.adam(schedule)
state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=optimizer)

In [ ]:
# Training loop
batch_size = 256
n_steps = 8000
losses = []

key = jr.PRNGKey(1)
for step in range(n_steps):
    key, subkey = jr.split(key)
    inputs, targets = generate_batch(subkey, batch_size, n_particles)
    state, loss = train_step(state, inputs, targets)
    losses.append(float(loss))
    if step % 2000 == 0:
        print(f"Step {step:5d}  Loss: {loss:.4f}")

print(f"Step {n_steps:5d}  Loss: {losses[-1]:.4f}")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3))
ax.semilogy(losses)
ax.set_xlabel("Step")
ax.set_ylabel("MSE Loss")
plt.tight_layout()
plt.show()

## 4. Attention Maps

Each head produces an $n \times n$ attention matrix. Let's see what they learned.

In [ ]:
# Generate a test example
key = jr.PRNGKey(123)
test_inputs, test_forces = generate_batch(key, 1, n_particles)
x_test = test_inputs[0]  # (n, 3)
f_test = test_forces[0]  # (n, 2)

# Forward pass with attention maps
preds, attn_maps = model.apply(state.params, x_test, return_attention=True)

print(
    f"Attention maps: {len(attn_maps)} layers, each ({attn_maps[0].shape[0]} heads, {attn_maps[0].shape[1]}x{attn_maps[0].shape[2]})"
)

In [ ]:
# Compute pairwise distances and mass products for comparison
pos_test = x_test[:, :2]
mass_test = x_test[:, 2]
dx = pos_test[None, :, :] - pos_test[:, None, :]
dists = jnp.sqrt(jnp.sum(dx**2, axis=-1) + 0.1**2)  # (n, n)
mass_prod = mass_test[:, None] * mass_test[None, :]  # (n, n)

# Force magnitude: m_i * m_j / r^2 (what attention *should* approximate)
force_mag = mass_prod / dists**2
# Zero diagonal
force_mag = force_mag * (1 - jnp.eye(n_particles))
# Normalize rows for comparison with attention
force_mag_norm = force_mag / force_mag.sum(axis=-1, keepdims=True)

In [ ]:
n_layers = len(attn_maps)
n_heads = attn_maps[0].shape[0]

fig, axes = plt.subplots(n_layers, n_heads + 1, figsize=(3 * (n_heads + 1), 3 * n_layers))
if n_layers == 1:
    axes = axes[None, :]

for layer in range(n_layers):
    for head in range(n_heads):
        ax = axes[layer, head]
        im = ax.imshow(attn_maps[layer][head], cmap="Blues", vmin=0)
        # Correlation with force magnitude
        A = np.array(attn_maps[layer][head]).flatten()
        F = np.array(force_mag_norm).flatten()
        corr = np.corrcoef(A, F)[0, 1]
        ax.set_title(f"L{layer + 1} H{head + 1} (r={corr:.2f})", fontsize=10)
        ax.set_xlabel("key $j$")
        if head == 0:
            ax.set_ylabel("query $i$")

    # Show normalized force magnitude for comparison
    ax = axes[layer, n_heads]
    ax.imshow(force_mag_norm, cmap="Reds", vmin=0)
    ax.set_title(r"$m_i m_j / r_{ij}^2$ (normalized)", fontsize=10)
    ax.set_xlabel("$j$")

plt.tight_layout()
plt.show()

## 5. Predictions vs. Ground Truth

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for ax, F, title in zip(axes, [f_test, preds], ["Ground truth", "Predicted"]):
    ax.scatter(pos_test[:, 0], pos_test[:, 1], s=mass_test * 200, c="steelblue", edgecolors="k", zorder=5)
    ax.quiver(pos_test[:, 0], pos_test[:, 1], F[:, 0], F[:, 1], color="firebrick", scale=30, width=0.006, zorder=4)
    for i in range(n_particles):
        ax.annotate(f"{i}", (pos_test[i, 0] + 0.05, pos_test[i, 1] + 0.05), fontsize=10, color="gray")
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.3)
    ax.set_aspect("equal")
    ax.set_title(title)

plt.tight_layout()
plt.show()